# BrainiacLLM on AWS (GPU) -- long-context, budget-aware, resumable
## Custom from-scratch tokenizer · packed long-sequence training · RoPE · KV-cache generation

**What this version changes compared with the Colab-TPU notebook**

1. **AWS GPU instead of TPU.** No `torch_xla`/libtpu at all (that was the source of the PJRT error). Plain PyTorch + CUDA, `bf16` autocast, fused AdamW, FlashAttention through `scaled_dot_product_attention`.
2. **Long generation.** Learned position embeddings are replaced by **RoPE**, the data is **packed** (documents are joined with `<eos>` and cut into full blocks, so no compute is wasted on padding and *every* position up to the longest context is really trained), and training runs in **stages of growing context**: `1024 -> 4096 -> 8192` tokens. 8192 tokens is roughly 6,000 words. Generation uses a **KV cache** (fast) and a sliding window, so it can keep writing past the trained context.
3. **Sized for a $100 budget.** `BUDGET_USD`, `RESERVE_USD` and `PRICE_PER_HOUR` in the config cell are used to *measure* your GPU's real speed and then compute how many tokens/documents you can afford. The run stops itself before the money runs out.
4. **Survives interruptions.** Time-based checkpoints, optional upload to S3, automatic restore from S3 on a new instance, and a watcher that saves immediately when AWS sends a **Spot interruption notice**. Data position is restored from the dataset's own state (no re-reading millions of documents).
5. Architecture fixes: pre-LayerNorm blocks (stable), tied input/output embeddings, causal FlashAttention (no `S x S` score matrix), chunked loss (no giant logits tensor), correct next-token shift.

---
## AWS setup (one time, ~10 minutes)

| Item | Recommendation |
|---|---|
| Instance | **g5.xlarge** (1x NVIDIA A10G 24 GB). About **$1.006/h on-demand**; Spot is usually **$0.45-0.65/h** (check the console). g6e.xlarge (L40S 48 GB) is faster but about 1.8x the price. |
| Image (AMI) | *Deep Learning OSS Nvidia Driver AMI GPU PyTorch 2.x (Ubuntu)* -- torch, CUDA and the AWS CLI are preinstalled |
| Disk | 150 GB gp3 (about $12/month, so only a few dollars for a week) |
| Role | IAM instance role with read/write to your S3 bucket (only if you use `S3_URI`) |
| Access | `ssh -L 8888:localhost:8888 ubuntu@<ip>` then `jupyter lab --no-browser` on the instance and open `localhost:8888` |

**Run it unattended (recommended for an 80+ hour job):** a browser tab can disconnect, a `tmux` session cannot.
```
jupyter nbconvert --to script BrainiacLLM_AWS_longctx.ipynb
tmux new -s train
python BrainiacLLM_AWS_longctx.py 2>&1 | tee train.log      # detach: Ctrl-b d
```
This notebook has no `!`/`%` magics, so it converts cleanly.

**Cost safety**
- Set `PRICE_PER_HOUR` to what you really pay. The plan is sized from it, and the run stops itself when estimated GPU spend reaches `BUDGET_USD - RESERVE_USD`.
- The instance **keeps billing after training ends**. Set `SHUTDOWN_WHEN_DONE = True`, or stop/terminate it yourself.
- After a Spot interruption just launch a new instance, set the same `S3_URI`, and **Run all**: it restores and continues.

## 1. Setup and Dependencies

In [ ]:
import subprocess, sys, os, importlib.util

# Must be set before CUDA is initialised: reduces fragmentation when the sequence length changes between stages.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

def _pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

for _mod, _pkg in [("datasets", "datasets"), ("tqdm", "tqdm"), ("matplotlib", "matplotlib")]:
    if importlib.util.find_spec(_mod) is None:
        _pip(_pkg)

import time, math, json, re, heapq, copy, queue, threading, shutil
import multiprocessing as mp
from pathlib import Path
from collections import Counter
from typing import List, Tuple, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint as torch_checkpoint
from tqdm.auto import tqdm
from datasets import load_dataset, interleave_datasets

print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())

## 2. GPU Device Setup

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if device.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    try:   # nvidia-smi instead of torch.cuda.get_device_name: keeps CUDA un-initialised until after the tokenizer's worker processes fork
        print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
                             capture_output=True, text=True).stdout.strip())
    except Exception:
        pass
else:
    print("WARNING: no CUDA GPU found. Launch a GPU instance (g5.xlarge) -- training on CPU is far too slow.")

USE_BF16 = device.type == "cuda"      # A10G / L4 / L40S / A100 all support bf16 autocast
print("device:", device, "| bf16 autocast:", USE_BF16)

## 3. Configuration (model, long-context stages, budget)

In [ ]:
class Config:
    # ------------------------- Model architecture -------------------------
    # ~110M parameters (GPT-2-small class) with tied embeddings. If the plan cell later reports
    # more than ~60 tokens per parameter, your budget can afford a bigger model, e.g.
    #   embedding_dim=1024, num_heads=16, num_layers=16  (~235M)   -- head_dim must stay 64
    vocab_size = 32000            # target tokenizer size (32k keeps ~1.3 tokens/word, so 8192 tokens ~ 6000 words)
    embedding_dim = 768
    num_heads = 12                # head_dim = 64
    num_layers = 12
    ff_dim = embedding_dim * 4
    max_seq_length = 8192         # longest context the model is trained at AND generates at (RoPE, no position table)
    rope_theta = 10000.0
    dropout = 0.0

    # ------------------------- Optimisation -------------------------
    learning_rate = 6e-4          # peak; warmup then cosine decay down to min_lr_ratio * peak
    min_lr_ratio = 0.1
    warmup_steps = 300
    weight_decay = 0.1
    beta2 = 0.95
    gradient_clip = 1.0
    tokens_per_step = 262_144     # tokens per optimizer step (constant across stages)
    micro_tokens = 32_768         # tokens per forward/backward pass: 32 x 1024, 8 x 4096 or 4 x 8192
    # (context length, share of the run). Early steps are cheap short-context, the last 35% teaches long text.
    seq_stages = [(1024, 0.65), (4096, 0.20), (8192, 0.15)]

    # ------------------------- Data -------------------------
    dataset_mixture = [
        {
            "name": "HuggingFaceFW/fineweb",
            "config": "CC-MAIN-2024-10",
            "split": "train",
            "text_field": "text",
            "weight": 0.20,
        },
        {
            "name": "HuggingFaceFW/finetranslations",
            "config": "all",
            "split": "train",
            "text_field": "og_full_text",
            "weight": 0.15,
        },
        {
            "name": "wikimedia/wikipedia",
            "config": "20231101.en",
            "split": "train",
            "text_field": "text",
            "weight": 0.15,
        },
        {
            "name": "HuggingFaceFW/fineweb-edu",
            "config": "CC-MAIN-2024-10",
            "split": "train",
            "text_field": "text",
            "weight": 0.20,
        },
        {
            "name": "open-web-math/open-web-math",
            "config": None,
            "split": "train",
            "text_field": "text",
            "weight": 0.15,
        },
        {
            "data_files": "hf://datasets/codeparrot/github-code-clean/data/*.parquet",
            "split": "train",
            "text_field": "code",
            "weight": 0.15,
        },
    ]
    num_tokenizer_docs = 100000   # documents used to learn the tokenizer
    max_doc_chars = 100_000       # a single document is cut here before tokenising (huge code files / pages)

    # ------------------------- Checkpoints / paths -------------------------
    checkpoint_every_minutes = 20     # time based: a Spot reclaim loses at most this much work
    model_name = "brainiacLLM_aws"
    save_dir = str(Path.home() / "brainiac_runs" / model_name)

config = Config()

# ============================================================
# BUDGET  (all money in USD)
# ============================================================
BUDGET_USD = 100.0          # everything you are willing to spend on this project
RESERVE_USD = 15.0          # kept back for EBS disk, S3, tokenizer training, benchmarks, test generation, idle time
PRICE_PER_HOUR = 1.006      # g5.xlarge on-demand. If you use Spot, put the Spot price you see (about 0.45-0.65):
                            # the plan then buys more tokens for the same money.
TRAIN_BUDGET_USD = BUDGET_USD - RESERVE_USD
PLAN_EFFICIENCY = 0.90      # assume 90% of the benchmarked speed (data loading, optimizer step, checkpoint saves)

S3_URI = ""                 # e.g. "s3://my-bucket/brainiac/brainiacLLM_aws"  ("" = local disk only)
SHUTDOWN_WHEN_DONE = False  # True = `sudo shutdown -h now` after the final save, so you stop paying for the GPU

assert config.tokens_per_step % config.micro_tokens == 0
assert all(config.micro_tokens % s == 0 and s <= config.max_seq_length for s, _ in config.seq_stages)
assert abs(sum(f for _, f in config.seq_stages) - 1.0) < 1e-6
assert config.embedding_dim % config.num_heads == 0

print("Model Config:")
for k in ["vocab_size", "embedding_dim", "num_heads", "num_layers", "max_seq_length", "learning_rate",
          "warmup_steps", "tokens_per_step", "micro_tokens", "seq_stages", "checkpoint_every_minutes", "save_dir"]:
    print(f"  {k}: {getattr(config, k)}")
print(f"Budget: ${BUDGET_USD:.0f} total = ${TRAIN_BUDGET_USD:.0f} GPU training at ${PRICE_PER_HOUR}/h "
      f"= up to {TRAIN_BUDGET_USD / PRICE_PER_HOUR:.0f} GPU-hours (+ ${RESERVE_USD:.0f} reserve)")

## 3b. Resume Settings

You normally do **not** need to edit anything here.

- No checkpoint in `config.save_dir` (and none in S3) -> **fresh run**: trains the tokenizer, then benchmarks your GPU and writes `run_plan.json`.
- A checkpoint exists -> **resume automatically**: loads tokenizer, weights, optimizer, step counter, dataset position and the *same plan file* (so the learning-rate schedule and stage boundaries never change).
- `FORCE_FRESH = True` ignores existing checkpoints (files are not deleted).

Do **not** change the config, `tokens_per_step` or the budget between sessions of the same run.

In [ ]:
FORCE_FRESH = False
RESUME_DIR = config.save_dir
os.makedirs(RESUME_DIR, exist_ok=True)

CHECKPOINT_CANDIDATES = ["checkpoint_latest.pt", "final_model.pt"]
PROGRESS_FILE = os.path.join(config.save_dir, "progress.json")
PLAN_FILE = os.path.join(config.save_dir, "run_plan.json")


# ---------- optional S3 sync (uses the AWS CLI + the instance role) ----------
def s3_upload(path):
    if not S3_URI or not os.path.exists(path):
        return False
    if shutil.which("aws") is None:
        print("  (aws CLI not found -- skipping S3 upload)")
        return False
    r = subprocess.run(["aws", "s3", "cp", path, f"{S3_URI.rstrip('/')}/{os.path.basename(path)}", "--only-show-errors"])
    return r.returncode == 0


def s3_restore():
    if not S3_URI or shutil.which("aws") is None:
        return False
    print(f"Looking for a checkpoint in {S3_URI} ...")
    r = subprocess.run(["aws", "s3", "sync", S3_URI.rstrip("/") + "/", config.save_dir, "--only-show-errors"])
    return r.returncode == 0


def have_checkpoint():
    return any(os.path.exists(os.path.join(RESUME_DIR, f)) for f in CHECKPOINT_CANDIDATES)


if not FORCE_FRESH and not have_checkpoint() and S3_URI:
    s3_restore()

RESUME = (not FORCE_FRESH) and have_checkpoint()
if RESUME:
    print(f"RESUME=True  -> checkpoint found in {RESUME_DIR}")
else:
    print("RESUME=False -> fresh run")


def load_progress():
    if os.path.exists(PROGRESS_FILE):
        with open(PROGRESS_FILE) as f:
            return json.load(f)
    return {}


def record_progress(filename, info):
    """Human-readable log of the checkpoints (the real counters live inside each checkpoint). Atomic write."""
    prog = load_progress()
    prog[filename] = dict(info, saved=time.ctime())
    tmp = PROGRESS_FILE + ".tmp"
    with open(tmp, "w") as f:
        json.dump(prog, f, indent=2)
    os.replace(tmp, PROGRESS_FILE)

## 4. Load Mixed Dataset (general knowledge + math + code, streaming)

In [ ]:
def build_mixed_dataset(mixture, seed=42, skip=0):
    """Interleave several HF streaming datasets into one text stream, sampled according to each
    source's `weight`. Every source is normalised to a single "text" column first."""
    streams, probs = [], []
    for spec in mixture:
        if "data_files" in spec:
            ds = load_dataset("parquet", data_files=spec["data_files"], split=spec["split"], streaming=True)
        else:
            ds = load_dataset(spec["name"], name=spec.get("config"), split=spec["split"], streaming=True)
        field = spec["text_field"]
        if field != "text":
            ds = ds.map(lambda ex, f=field: {"text": ex[f]})
        if ds.column_names:
            drop_cols = [c for c in ds.column_names if c != "text"]
            if drop_cols:
                ds = ds.remove_columns(drop_cols)
        streams.append(ds)
        probs.append(spec["weight"])

    total = sum(probs)
    probs = [p / total for p in probs]

    mixed = interleave_datasets(streams, probabilities=probs, seed=seed, stopping_strategy="all_exhausted")
    if skip:
        mixed = mixed.skip(skip)
    return mixed


print("build_mixed_dataset() defined")

In [ ]:
# Optional: a Hugging Face token avoids anonymous rate limits while streaming 6 datasets.
#   export HF_TOKEN=hf_xxx   (before starting jupyter)   or uncomment the next line:
# os.environ["HF_TOKEN"] = "hf_..."

print("Loading mixed dataset (general knowledge + math + code)...")
dataset = build_mixed_dataset(config.dataset_mixture)

sample_iter = iter(dataset)
sample = next(sample_iter)
print(f"Sample text (first 300 chars):\n{sample['text'][:300]}...")

## 5. Custom Byte-Level BPE Tokenizer (from scratch, no `tokenizers`/`transformers`)

Same algorithm as GPT-2's BPE: regex pre-tokenisation into words, merges learned over *unique word counts*, incremental pair counting with a heap, a per-word encode cache, and a 256-byte base vocabulary (any UTF-8 text encodes, no `<unk>` needed). Word counting uses worker processes on Linux (safe here because there is no JAX/XLA thread pool any more).

In [ ]:
PRETOKEN_PATTERN = re.compile(
    r"'s|'t|'re|'ve|'m|'ll|'d| ?[a-zA-Z]+| ?[0-9]+| ?[^\sa-zA-Z0-9]+|\s+"
)


def _count_words_chunk(texts_chunk):
    """Top-level (picklable) so it can run in a worker process."""
    local_counts = Counter()
    for text in texts_chunk:
        for w in PRETOKEN_PATTERN.findall(text):
            local_counts[w] += 1
    return local_counts


class ByteBPETokenizer:
    """Fast from-scratch byte-level BPE tokenizer (word-level pretokenisation + incremental heap pair counting)."""

    def __init__(self):
        self.merges = {}                                   # (int, int) -> int
        self.vocab = {i: bytes([i]) for i in range(256)}   # id -> bytes
        self.special_tokens = {}                           # name -> id
        self.special_tokens_inv = {}                       # id -> name
        self._encode_cache = {}                            # word str -> list[int]

    @staticmethod
    def _pretokenize(text):
        return PRETOKEN_PATTERN.findall(text)

    @staticmethod
    def _merge(ids, pair, new_id):
        new_ids = []
        i = 0
        n = len(ids)
        while i < n:
            if i < n - 1 and ids[i] == pair[0] and ids[i + 1] == pair[1]:
                new_ids.append(new_id)
                i += 2
            else:
                new_ids.append(ids[i])
                i += 1
        return new_ids

    # ---------- training ----------
    def train(self, texts: List[str], vocab_size: int, verbose: bool = True,
              min_word_freq: int = 2, num_workers: int = None):
        assert vocab_size >= 256

        num_workers = num_workers or max(1, min(8, (os.cpu_count() or 1) - 1))
        word_freq = Counter()
        if num_workers > 1 and len(texts) > num_workers:
            chunk_size = -(-len(texts) // num_workers)
            chunks = [texts[i:i + chunk_size] for i in range(0, len(texts), chunk_size)]
            try:
                with mp.get_context("fork").Pool(num_workers) as pool:
                    for partial in pool.imap_unordered(_count_words_chunk, chunks):
                        word_freq.update(partial)
            except Exception as e:
                if verbose:
                    print(f"  multiprocessing unavailable ({e}) -- falling back to single-process counting")
                word_freq = Counter()
                for text in texts:
                    for w in self._pretokenize(text):
                        word_freq[w] += 1
        else:
            for text in texts:
                for w in self._pretokenize(text):
                    word_freq[w] += 1
        if verbose:
            print(f"  {len(word_freq)} unique words from {len(texts)} documents ({num_workers} worker(s))")

        if min_word_freq > 1:
            before = len(word_freq)
            word_freq = Counter({w: c for w, c in word_freq.items() if c >= min_word_freq})
            if verbose:
                print(f"  dropped {before - len(word_freq)} words seen < {min_word_freq}x -- {len(word_freq)} remain")

        word_ids = {w: list(w.encode("utf-8")) for w in word_freq}

        pair_counts = Counter()
        pair_index = {}
        for w, freq in word_freq.items():
            ids = word_ids[w]
            for a, b in zip(ids, ids[1:]):
                pair_counts[(a, b)] += freq
                pair_index.setdefault((a, b), set()).add(w)

        heap = [(-c, p) for p, c in pair_counts.items()]
        heapq.heapify(heap)

        num_merges = vocab_size - 256
        merges_done = 0
        while merges_done < num_merges and heap:
            neg_count, pair = heapq.heappop(heap)
            if pair_counts.get(pair, 0) != -neg_count:
                continue
            if -neg_count < 2:
                break

            new_id = 256 + merges_done
            for w in pair_index.pop(pair, ()):
                freq = word_freq[w]
                ids = word_ids[w]
                for a, b in zip(ids, ids[1:]):
                    pair_counts[(a, b)] -= freq
                new_ids = self._merge(ids, pair, new_id)
                word_ids[w] = new_ids
                for a, b in zip(new_ids, new_ids[1:]):
                    pair_counts[(a, b)] = pair_counts.get((a, b), 0) + freq
                    pair_index.setdefault((a, b), set()).add(w)
                    heapq.heappush(heap, (-pair_counts[(a, b)], (a, b)))

            self.merges[pair] = new_id
            self.vocab[new_id] = self.vocab[pair[0]] + self.vocab[pair[1]]
            pair_counts.pop(pair, None)
            merges_done += 1

            if verbose and merges_done % 2000 == 0:
                print(f"  merge {merges_done}/{num_merges}")

        if verbose:
            print(f"  done: {merges_done}/{num_merges} merges completed")

        for name in ["<pad>", "<bos>", "<eos>", "<unk>"]:
            new_id = len(self.vocab)
            self.special_tokens[name] = new_id
            self.special_tokens_inv[new_id] = name
            self.vocab[new_id] = name.encode("utf-8")

    # ---------- encode / decode ----------
    def _bpe_word(self, word: str) -> List[int]:
        if word in self._encode_cache:
            return self._encode_cache[word]
        ids = list(word.encode("utf-8"))
        while len(ids) >= 2:
            pairs = zip(ids, ids[1:])
            ranked = [(self.merges[p], p) for p in pairs if p in self.merges]
            if not ranked:
                break
            _, best_pair = min(ranked)
            ids = self._merge(ids, best_pair, self.merges[best_pair])
        if len(self._encode_cache) >= 2_000_000:
            self._encode_cache.clear()
        self._encode_cache[word] = ids
        return ids

    def encode(self, text: str) -> List[int]:
        ids = []
        for w in self._pretokenize(text):
            ids.extend(self._bpe_word(w))
        return ids

    def decode(self, ids: List[int]) -> str:
        parts = []
        for i in ids:
            if i in self.special_tokens_inv:
                continue
            parts.append(self.vocab.get(i, b""))
        return b"".join(parts).decode("utf-8", errors="replace")

    @property
    def vocab_size(self):
        return len(self.vocab)

    # ---------- persistence ----------
    def save(self, path: str):
        os.makedirs(path, exist_ok=True)
        merges_serializable = {f"{a},{b}": idx for (a, b), idx in self.merges.items()}
        with open(os.path.join(path, "tokenizer.json"), "w") as f:
            json.dump({"merges": merges_serializable, "special_tokens": self.special_tokens}, f)

    @classmethod
    def load(cls, path: str):
        tok = cls()
        with open(os.path.join(path, "tokenizer.json")) as f:
            data = json.load(f)
        tok.merges = {}
        for k, idx in data["merges"].items():
            a, b = k.split(",")
            tok.merges[(int(a), int(b))] = idx
            tok.vocab[idx] = tok.vocab[int(a)] + tok.vocab[int(b)]
        tok.special_tokens = data["special_tokens"]
        for name, idx in tok.special_tokens.items():
            tok.special_tokens_inv[idx] = name
            tok.vocab[idx] = name.encode("utf-8")
        return tok


print("ByteBPETokenizer defined!")

In [ ]:
if RESUME:
    tok_dir = os.path.join(RESUME_DIR, "tokenizer")
    if not os.path.exists(os.path.join(tok_dir, "tokenizer.json")):
        raise FileNotFoundError(f"RESUME=True but no tokenizer found at {tok_dir}. "
                                f"Check RESUME_DIR / S3_URI, or set FORCE_FRESH=True for a fresh run.")
    print(f"Loading saved tokenizer from {tok_dir} (not retraining)...")
    tokenizer = ByteBPETokenizer.load(tok_dir)

    cfg_path = os.path.join(RESUME_DIR, "config.json")
    if os.path.exists(cfg_path):
        with open(cfg_path) as f:
            saved_cfg = json.load(f)
        for k, v in saved_cfg.items():
            if k != "model_name":
                setattr(config, k, v)
        print("Architecture settings restored from config.json")
else:
    print("Preparing tokenizer training corpus (general knowledge + math + code)...")
    tokenizer_corpus = []
    for idx, example in enumerate(dataset):
        if idx >= config.num_tokenizer_docs:
            break
        tokenizer_corpus.append(example["text"][:20000])
    print(f"Tokenizer training corpus: {len(tokenizer_corpus)} documents")

    print(f"Training custom BPE tokenizer (pure Python -- a few minutes for vocab_size={config.vocab_size})...")
    tokenizer = ByteBPETokenizer()
    tokenizer.train(tokenizer_corpus, vocab_size=config.vocab_size)
    del tokenizer_corpus

config.vocab_size = tokenizer.vocab_size      # lock in the real vocab size (learned merges + 4 specials)
print(f"Tokenizer ready! Final vocab size: {tokenizer.vocab_size}")

PAD_ID = tokenizer.special_tokens["<pad>"]
BOS_ID = tokenizer.special_tokens["<bos>"]
EOS_ID = tokenizer.special_tokens["<eos>"]


def save_config_json():
    with open(os.path.join(config.save_dir, "config.json"), "w") as f:
        json.dump({
            "vocab_size": config.vocab_size, "embedding_dim": config.embedding_dim,
            "num_heads": config.num_heads, "num_layers": config.num_layers,
            "ff_dim": config.ff_dim, "max_seq_length": config.max_seq_length,
            "rope_theta": config.rope_theta, "model_name": config.model_name,
        }, f, indent=2)


os.makedirs(config.save_dir, exist_ok=True)
tokenizer.save(os.path.join(config.save_dir, "tokenizer"))
save_config_json()
print(f"Tokenizer + config saved to {config.save_dir}")

In [ ]:
# How many tokens is a document / a word? (used to turn the token budget into "documents" and "words")
_sample_docs = []
for _i, _ex in enumerate(build_mixed_dataset(config.dataset_mixture, seed=7)):
    if _i >= 300:
        break
    _sample_docs.append(_ex["text"][:config.max_doc_chars])

AVG_TOKENS_PER_DOC = float(np.mean([len(tokenizer.encode(t)) + 2 for t in _sample_docs]))

# words -> tokens on prose-like documents only (code / LaTeX would distort it)
_prose = [t for t in _sample_docs if t and sum(ch.isalpha() or ch == " " for ch in t[:3000]) / len(t[:3000]) > 0.85]
_tok_prose = sum(len(tokenizer.encode(t[:3000])) for t in _prose)
_words_prose = sum(len(t[:3000].split()) for t in _prose)
TOKENS_PER_WORD = _tok_prose / max(1, _words_prose)

print(f"Average document = {AVG_TOKENS_PER_DOC:,.0f} tokens (cut at {config.max_doc_chars:,} chars)")
print(f"Prose: ~{TOKENS_PER_WORD:.2f} tokens per word  ->  {config.max_seq_length} tokens ~ "
      f"{int(config.max_seq_length / TOKENS_PER_WORD):,} words of context")
del _sample_docs, _prose

## 6. Packed Streaming Data (no padding)

Documents are tokenised on the fly in a **background thread**, joined as `<bos> doc <eos> <bos> doc <eos> ...` and cut into blocks of `seq_len + 1` tokens (consecutive blocks overlap by one token, so no next-token target is lost). Every position of every block is real text, so long-context stages train the whole length.

For resuming, each batch carries the dataset's own `state_dict()` (shard + offset), which is restored on the next session instead of re-reading millions of documents; if that is not supported by your `datasets` version it falls back to `.skip(n)`.

In [ ]:
class PackedStream:
    """Background-thread producer of packed batches.
    Yields (tokens[micro_batch, seq_len+1] int64, docs_consumed_since_start, dataset_state_or_None)."""

    def __init__(self, mixture, tokenizer, seq_len, micro_batch, bos_id, eos_id,
                 data_state=None, skip_docs=0, shuffle_buffer=10000, seed=42,
                 max_doc_chars=100_000, prefetch=6, dataset_fn=None):
        self.mixture, self.tokenizer = mixture, tokenizer
        self.seq_len, self.micro_batch = seq_len, micro_batch
        self.bos_id, self.eos_id = bos_id, eos_id
        self.data_state, self.skip_docs = data_state, skip_docs
        self.shuffle_buffer, self.seed = shuffle_buffer, seed
        self.max_doc_chars = max_doc_chars
        self.dataset_fn = dataset_fn or build_mixed_dataset
        self.q = queue.Queue(maxsize=prefetch)
        self._stop = threading.Event()
        self._thread = threading.Thread(target=self._run, daemon=True)
        self._thread.start()

    def _put(self, item):
        while not self._stop.is_set():
            try:
                self.q.put(item, timeout=0.5)
                return True
            except queue.Full:
                continue
        return False

    def _run(self):
        try:
            ds = self.dataset_fn(self.mixture, seed=self.seed)
            if self.shuffle_buffer:
                ds = ds.shuffle(buffer_size=self.shuffle_buffer, seed=self.seed)   # windowed shuffle
            resumed = False
            if self.data_state is not None:
                try:
                    ds.load_state_dict(self.data_state)
                    resumed = True
                    print("  [data] restored the dataset position from the checkpoint (instant resume)")
                except Exception as e:
                    print(f"  [data] could not restore dataset state ({type(e).__name__}); falling back to skip({self.skip_docs:,})")
            if not resumed and self.skip_docs:
                ds = ds.skip(self.skip_docs)
            has_state = hasattr(ds, "state_dict")

            S, B = self.seq_len, self.micro_batch
            limit = self.max_doc_chars
            buf, rows, docs = [], [], 0
            for ex in ds:
                if self._stop.is_set():
                    return
                text = ex["text"]
                if not text:
                    continue
                truncated = len(text) > limit
                if truncated:
                    text = text[:limit]
                ids = [self.bos_id] + self.tokenizer.encode(text) + ([] if truncated else [self.eos_id])
                buf.extend(ids)
                docs += 1
                while len(buf) >= S + 1:
                    rows.append(buf[:S + 1])
                    del buf[:S]                      # keep the last token: it is the first token of the next block
                    if len(rows) == B:
                        arr = torch.from_numpy(np.asarray(rows, dtype=np.int64))
                        rows = []
                        state = None
                        if has_state:
                            try:
                                state = copy.deepcopy(ds.state_dict())
                            except Exception:
                                has_state = False
                        if not self._put(("batch", arr, docs, state)):
                            return
            self._put(("end", None, docs, None))
        except BaseException as e:           # surfaced in the training thread
            self._put(("error", e, 0, None))

    def __iter__(self):
        while True:
            try:
                kind, a, docs, state = self.q.get(timeout=1.0)
            except queue.Empty:
                if not self._thread.is_alive() and self.q.empty():
                    return
                continue
            if kind == "batch":
                yield a, docs, state
            elif kind == "end":
                return
            else:
                raise RuntimeError(f"data thread failed: {a!r}") from a

    def close(self):
        self._stop.set()
        try:
            while True:
                self.q.get_nowait()
        except queue.Empty:
            pass
        self._thread.join(timeout=5)


print("PackedStream defined!")

## 7. BrainiacLLM architecture (RoPE, FlashAttention, KV cache)

- **RoPE** rotary positions: no learned position table, so the context length is a training choice and not a parameter-count cost.
- **Pre-LayerNorm** residual blocks (much more stable than post-LN at 12+ layers), no biases, tied input/output embeddings.
- **Causal FlashAttention** through `F.scaled_dot_product_attention` (never builds the `S x S` score matrix).
- **KV-cache `generate()`**: one token costs one small forward pass instead of re-running the whole context; repetition penalty is vectorised and applies to a recent window only (penalising *every* earlier token, as the old version did, ruins long outputs).

In [ ]:
def precompute_rope(head_dim, max_len, theta):
    inv_freq = 1.0 / (theta ** (torch.arange(0, head_dim, 2).float() / head_dim))
    t = torch.arange(max_len).float()
    freqs = torch.outer(t, inv_freq)                 # (max_len, head_dim/2)
    return freqs.cos(), freqs.sin()


def apply_rope(x, cos, sin):
    # x: (B, H, T, head_dim); cos/sin: (T, head_dim/2)   -- rotate in fp32, return in x's dtype
    xf = x.float()
    x1, x2 = xf[..., : xf.shape[-1] // 2], xf[..., xf.shape[-1] // 2:]
    out = torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)
    return out.to(x.dtype)


class MultiHeadAttention(nn.Module):
    def __init__(self, embedding_dim: int, num_heads: int, dropout: float = 0.0):
        super().__init__()
        assert embedding_dim % num_heads == 0, "embedding_dim must be divisible by num_heads"
        self.embedding_dim = embedding_dim
        self.num_heads = num_heads
        self.head_dim = embedding_dim // num_heads
        self.dropout_p = dropout
        self.qkv = nn.Linear(embedding_dim, 3 * embedding_dim, bias=False)
        self.fc_out = nn.Linear(embedding_dim, embedding_dim, bias=False)

    def forward(self, x, cos, sin, past=None, use_cache=False):
        B, T, _ = x.shape
        q, k, v = self.qkv(x).split(self.embedding_dim, dim=-1)
        q = q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)

        if past is not None:
            k = torch.cat([past[0], k], dim=2)
            v = torch.cat([past[1], v], dim=2)
        present = (k, v) if use_cache else None

        dp = self.dropout_p if self.training else 0.0
        if past is None:
            out = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=dp)
        elif T == 1:
            out = F.scaled_dot_product_attention(q, k, v)            # one new token sees the whole cache
        else:
            S = k.size(2)
            mask = torch.ones(T, S, dtype=torch.bool, device=x.device).tril(diagonal=S - T)
            out = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
        out = out.transpose(1, 2).contiguous().view(B, T, self.embedding_dim)
        return self.fc_out(out), present


class FeedForward(nn.Module):
    def __init__(self, embedding_dim: int, ff_dim: int, dropout: float = 0.0):
        super().__init__()
        self.fc1 = nn.Linear(embedding_dim, ff_dim, bias=False)
        self.fc2 = nn.Linear(ff_dim, embedding_dim, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.fc2(self.dropout(F.gelu(self.fc1(x))))


class TransformerBlock(nn.Module):
    def __init__(self, embedding_dim: int, num_heads: int, ff_dim: int, dropout: float = 0.0):
        super().__init__()
        self.attention = MultiHeadAttention(embedding_dim, num_heads, dropout)
        self.feed_forward = FeedForward(embedding_dim, ff_dim, dropout)
        self.norm1 = nn.LayerNorm(embedding_dim)
        self.norm2 = nn.LayerNorm(embedding_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, cos, sin, past=None, use_cache=False):
        a, present = self.attention(self.norm1(x), cos, sin, past, use_cache)
        x = x + self.dropout(a)
        x = x + self.dropout(self.feed_forward(self.norm2(x)))
        return (x, present) if use_cache else x


class BrainiacLLM(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.grad_ckpt = False                                   # switched on per stage if memory is tight
        head_dim = config.embedding_dim // config.num_heads
        self.padded_vocab = (config.vocab_size + 63) // 64 * 64  # multiple of 64 = tensor-core friendly

        self.token_embedding = nn.Embedding(self.padded_vocab, config.embedding_dim)
        self.dropout = nn.Dropout(config.dropout)
        self.transformer_blocks = nn.ModuleList([
            TransformerBlock(config.embedding_dim, config.num_heads, config.ff_dim, config.dropout)
            for _ in range(config.num_layers)
        ])
        self.norm = nn.LayerNorm(config.embedding_dim)
        self.lm_head = nn.Linear(config.embedding_dim, self.padded_vocab, bias=False)
        self.lm_head.weight = self.token_embedding.weight        # tied embeddings

        cos, sin = precompute_rope(head_dim, config.max_seq_length, config.rope_theta)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)
        self._init_weights()

    def _init_weights(self):
        for module in self.modules():
            if isinstance(module, (nn.Linear, nn.Embedding)):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)
        scaled = 0.02 / math.sqrt(2 * self.config.num_layers)   # GPT-2 residual scaling
        for name, p in self.named_parameters():
            if name.endswith("fc_out.weight") or name.endswith("fc2.weight"):
                nn.init.normal_(p, mean=0.0, std=scaled)

    def forward(self, input_ids, past_kv=None, use_cache=False, start_pos=0,
                return_hidden=False, last_only=False):
        B, T = input_ids.shape
        cos = self.rope_cos[start_pos:start_pos + T]
        sin = self.rope_sin[start_pos:start_pos + T]
        x = self.dropout(self.token_embedding(input_ids))

        presents = []
        for i, block in enumerate(self.transformer_blocks):
            if use_cache:
                x, pr = block(x, cos, sin, None if past_kv is None else past_kv[i], True)
                presents.append(pr)
            elif self.grad_ckpt and self.training:
                x = torch_checkpoint(block, x, cos, sin, use_reentrant=False)
            else:
                x = block(x, cos, sin)

        x = self.norm(x)
        if return_hidden:                       # training: loss is computed in chunks from the hidden states
            return x
        if last_only:
            x = x[:, -1:, :]
        logits = self.lm_head(x)
        return (logits, presents) if use_cache else logits

    @torch.no_grad()
    def generate(self, input_ids, max_new_tokens=512, temperature=0.8, top_k=50, top_p=0.95,
                 repetition_penalty=1.1, repetition_window=256, eos_id=None, stream=None):
        """Autoregressive generation with a KV cache and a sliding window.
        When the context window (config.max_seq_length) is full, the last half is kept and re-encoded,
        so the text can continue far beyond the trained length."""
        self.eval()
        dev = next(self.parameters()).device
        ctx_max = self.config.max_seq_length
        out = input_ids.to(dev)
        cur = out[:, -(ctx_max - 1):]
        past, pos = None, 0

        for _ in range(max_new_tokens):
            if past is not None and pos >= ctx_max:                  # window full: re-prefill with the last half
                cur, past, pos = out[:, -(ctx_max // 2):], None, 0
            with torch.autocast(dev.type, dtype=torch.bfloat16, enabled=(dev.type == "cuda")):
                logits, past = self(cur, past_kv=past, use_cache=True, start_pos=pos, last_only=True)
            pos += cur.shape[1]
            logits = logits[:, -1, : self.config.vocab_size].float()

            if repetition_penalty != 1.0 and repetition_window > 0:
                recent = out[:, -repetition_window:]
                score = logits.gather(1, recent)
                score = torch.where(score > 0, score / repetition_penalty, score * repetition_penalty)
                logits.scatter_(1, recent, score)

            if temperature <= 0:
                next_token = logits.argmax(dim=-1, keepdim=True)
            else:
                logits = logits / temperature
                if top_k and top_k > 0:
                    kth = torch.topk(logits, min(top_k, logits.size(-1)))[0][..., -1, None]
                    logits = logits.masked_fill(logits < kth, float("-inf"))
                if top_p is not None and top_p < 1.0:
                    sorted_logits, sorted_idx = torch.sort(logits, descending=True, dim=-1)
                    probs_sorted = F.softmax(sorted_logits, dim=-1)
                    remove = (probs_sorted.cumsum(dim=-1) - probs_sorted) > top_p
                    sorted_logits = sorted_logits.masked_fill(remove, float("-inf"))
                    logits = torch.full_like(logits, float("-inf")).scatter(1, sorted_idx, sorted_logits)
                next_token = torch.multinomial(F.softmax(logits, dim=-1), num_samples=1)

            out = torch.cat([out, next_token], dim=-1)
            if stream is not None:
                stream(next_token)
            if eos_id is not None and bool((next_token == eos_id).all()):
                break
            cur = next_token
        return out


print("BrainiacLLM (RoPE + FlashAttention + KV cache) defined!")

## 8. Initialize Model on GPU

In [ ]:
model = BrainiacLLM(config).to(device)

total_params = sum(p.numel() for p in model.parameters())          # tied weights are counted once
print("BrainiacLLM Model Summary:")
print(f"  Total parameters: {total_params:,}")
print(f"  Layers x dim x heads: {config.num_layers} x {config.embedding_dim} x {config.num_heads}")
print(f"  Model size: {total_params * 4 / 1e6:.0f} MB (fp32)")
print(f"  Longest context: {config.max_seq_length} tokens")

## 8b. Load Saved Weights (only when resuming)

In [ ]:
def torch_load_cpu(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")


resume_checkpoint = {}
if RESUME:
    found = [os.path.join(RESUME_DIR, f) for f in CHECKPOINT_CANDIDATES if os.path.exists(os.path.join(RESUME_DIR, f))]
    ckpt_path = max(found, key=os.path.getmtime)                     # newest file wins
    print("Checkpoint files found:")
    for p in found:
        print(f"  {os.path.basename(p):24s} {os.path.getsize(p) / 1e6:9.1f} MB")
    print(f"Using: {ckpt_path}")
    raw = torch_load_cpu(ckpt_path)
    if isinstance(raw, dict) and "model_state_dict" in raw:
        resume_checkpoint = raw
        model.load_state_dict(raw["model_state_dict"])
    else:
        model.load_state_dict(raw)
        raise KeyError("This file has weights only (no step/optimizer/plan state) -- it cannot continue a planned run.")
    del raw
    print(f"Weights loaded. Resuming at step {resume_checkpoint['global_step']:,}, "
          f"{resume_checkpoint['docs_seen']:,} documents seen, {resume_checkpoint['gpu_seconds'] / 3600:.2f} GPU-hours used.")
else:
    print("RESUME=False -> starting from randomly initialised weights.")

## 9. Chunked loss and the budget plan

`chunked_lm_loss` never materialises the full `(tokens x vocab)` logits (that alone would be 4 GB at 32k tokens): logits are produced `chunk` rows at a time and recomputed in backward.

**The plan cell** (fresh runs only) measures your GPU's real tokens/second at every context length, then solves for the number of tokens that fits `TRAIN_BUDGET_USD / PRICE_PER_HOUR` GPU-hours. The result (steps, stage boundaries, expected documents) is saved to `run_plan.json` and re-used unchanged on every resume.

In [ ]:
def chunked_lm_loss(hidden, lm_head, labels, chunk=4096):
    """Mean next-token cross entropy. hidden: (B, S, D) from model(..., return_hidden=True);
    labels: (B, S) already shifted (labels[:, i] is the token that follows position i)."""
    h = hidden.reshape(-1, hidden.size(-1))
    y = labels.reshape(-1)

    def _chunk_loss(h_c, y_c):
        return F.cross_entropy(lm_head(h_c).float(), y_c, reduction="sum")

    total = h.new_zeros((), dtype=torch.float32)
    for i in range(0, h.size(0), chunk):
        total = total + torch_checkpoint(_chunk_loss, h[i:i + chunk], y[i:i + chunk], use_reentrant=False)
    return total / y.numel()


def _sync():
    if device.type == "cuda":
        torch.cuda.synchronize()


def run_benchmark(seq_len, steps=4, warmup=2):
    mb = config.micro_tokens // seq_len
    x = torch.randint(0, config.vocab_size, (mb, seq_len + 1), device=device)
    model.train()

    def one():
        with torch.autocast(device.type, dtype=torch.bfloat16, enabled=USE_BF16):
            h = model(x[:, :-1], return_hidden=True)
            loss = chunked_lm_loss(h, model.lm_head, x[:, 1:])
        loss.backward()

    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    for _ in range(warmup):
        one()
    _sync()
    t0 = time.time()
    for _ in range(steps):
        one()
    _sync()
    dt = time.time() - t0
    peak = torch.cuda.max_memory_allocated() / 1e9 if device.type == "cuda" else 0.0
    model.zero_grad(set_to_none=True)
    return mb * seq_len * steps / dt, peak


def benchmark_stage(seq_len):
    """tokens/s, peak GB, used_grad_checkpointing  -- retries with gradient checkpointing on an out-of-memory error."""
    for ckpt in (False, True):
        model.grad_ckpt = ckpt
        try:
            tps, peak = run_benchmark(seq_len)
            return tps, peak, ckpt
        except torch.cuda.OutOfMemoryError:
            model.zero_grad(set_to_none=True)
            if device.type == "cuda":
                torch.cuda.empty_cache()
            print(f"  ctx {seq_len}: out of memory" + (" -> retrying with gradient checkpointing" if not ckpt else ""))
    raise RuntimeError(f"ctx {seq_len} does not fit in GPU memory even with gradient checkpointing; lower config.micro_tokens")

In [ ]:
if RESUME:
    if not os.path.exists(PLAN_FILE):
        raise FileNotFoundError(f"{PLAN_FILE} is missing -- it is saved next to the checkpoints (and in S3).")
    with open(PLAN_FILE) as f:
        plan = json.load(f)
    print(f"Loaded the saved plan: {plan['total_steps']:,} steps, {plan['total_tokens'] / 1e9:.2f}B tokens")
else:
    print("Benchmarking your GPU at each context length (about a minute) ...")
    tps_by_len, peak_by_len, ckpt_by_len = {}, {}, {}
    for S, frac in config.seq_stages:
        tps, peak, ck = benchmark_stage(S)
        tps_by_len[S], peak_by_len[S], ckpt_by_len[S] = tps, peak, ck
        print(f"  ctx {S:>5}: {tps:>9,.0f} tokens/s | peak memory {peak:5.1f} GB | grad-checkpointing: {ck}")
    model.grad_ckpt = False

    secs_per_token = sum(frac / tps_by_len[S] for S, frac in config.seq_stages)       # blended over the stages
    train_seconds = TRAIN_BUDGET_USD / PRICE_PER_HOUR * 3600
    total_tokens = train_seconds * PLAN_EFFICIENCY / secs_per_token
    total_steps = max(10, int(total_tokens // config.tokens_per_step) // 10 * 10)
    total_tokens = total_steps * config.tokens_per_step

    stage_plan, cum = [], 0
    for i, (S, frac) in enumerate(config.seq_stages):
        n = total_steps - cum if i == len(config.seq_stages) - 1 else int(round(total_steps * frac))
        cum += n
        stage_plan.append({"seq_len": S, "end_step": cum})

    plan = {
        "total_steps": total_steps, "total_tokens": total_tokens, "tokens_per_step": config.tokens_per_step,
        "stage_plan": stage_plan, "grad_ckpt": {str(k): v for k, v in ckpt_by_len.items()},
        "tokens_per_second": {str(k): v for k, v in tps_by_len.items()},
        "est_documents": int(total_tokens / AVG_TOKENS_PER_DOC), "params": total_params,
        "budget_usd": BUDGET_USD, "train_budget_usd": TRAIN_BUDGET_USD, "price_per_hour": PRICE_PER_HOUR,
        "created": time.ctime(),
    }
    with open(PLAN_FILE, "w") as f:
        json.dump(plan, f, indent=2)
    s3_upload(PLAN_FILE)

hours = sum((st["end_step"] - (plan["stage_plan"][i - 1]["end_step"] if i else 0)) * plan["tokens_per_step"]
            / plan["tokens_per_second"][str(st["seq_len"])] for i, st in enumerate(plan["stage_plan"])) / 3600
print("=" * 64)
print(f"RUN PLAN for ${plan['train_budget_usd']:.0f} of GPU time at ${plan['price_per_hour']}/h")
print(f"  parameters            : {plan['params'] / 1e6:,.0f} M")
print(f"  training tokens       : {plan['total_tokens'] / 1e9:.2f} B   ({plan['total_tokens'] / plan['params']:.0f} tokens per parameter)")
print(f"  training DOCUMENTS    : ~{plan['est_documents'] / 1e6:.1f} million   (~{AVG_TOKENS_PER_DOC:,.0f} tokens each)")
print(f"  optimizer steps       : {plan['total_steps']:,}  ({plan['tokens_per_step']:,} tokens each)")
for i, st in enumerate(plan["stage_plan"]):
    a = plan["stage_plan"][i - 1]["end_step"] if i else 0
    print(f"    steps {a:>7,} -> {st['end_step']:>7,}: context {st['seq_len']}")
print(f"  expected GPU time     : {hours:.1f} h  ->  ~${hours * plan['price_per_hour']:.0f} at the benchmarked speed")
print("=" * 64)
if plan["total_tokens"] / plan["params"] > 60:
    print("NOTE: > 60 tokens/parameter. Your budget can pay for a bigger model (see the note in the config cell); "
          "delete run_plan.json + the tokenizer folder and re-run with e.g. embedding_dim=1024, num_heads=16, num_layers=16.")
elif plan["total_tokens"] / plan["params"] < 10:
    print("NOTE: < 10 tokens/parameter: the model is large for this budget. Consider fewer layers or a lower price per hour.")

## 10. Optimizer and learning-rate schedule

In [ ]:
decay_params = [p for n, p in model.named_parameters() if p.requires_grad and p.dim() >= 2]
nodecay_params = [p for n, p in model.named_parameters() if p.requires_grad and p.dim() < 2]
optimizer = torch.optim.AdamW(
    [{"params": decay_params, "weight_decay": config.weight_decay},
     {"params": nodecay_params, "weight_decay": 0.0}],
    lr=config.learning_rate, betas=(0.9, config.beta2), eps=1e-8,
    fused=(device.type == "cuda"),
)

if RESUME and "optimizer_state_dict" in resume_checkpoint:
    optimizer.load_state_dict(resume_checkpoint["optimizer_state_dict"])
    print("Optimizer state restored from checkpoint")


def lr_at(step):
    """Linear warmup, then cosine decay to min_lr_ratio * peak over the WHOLE planned run (not per session)."""
    if step < config.warmup_steps:
        return config.learning_rate * (step + 1) / config.warmup_steps
    progress = min(1.0, (step - config.warmup_steps) / max(1, plan["total_steps"] - config.warmup_steps))
    cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
    return config.learning_rate * (config.min_lr_ratio + (1.0 - config.min_lr_ratio) * cosine)


train_state = {
    "global_step": resume_checkpoint.get("global_step", 0),
    "docs_seen": resume_checkpoint.get("docs_seen", 0),
    "gpu_seconds": resume_checkpoint.get("gpu_seconds", 0.0),
    "data_state": resume_checkpoint.get("data_state"),
}
loss_log = list(resume_checkpoint.get("loss_log", []))
del resume_checkpoint

print(f"Schedule: warmup {config.warmup_steps} steps -> cosine decay to {config.min_lr_ratio:.0%} of peak over {plan['total_steps']:,} steps")
print(f"Progress: step {train_state['global_step']:,} / {plan['total_steps']:,} "
      f"({100 * train_state['global_step'] / plan['total_steps']:.2f}%)   LR now {lr_at(train_state['global_step']):.2e}")

## 11. Training loop (staged context, time-based checkpoints, spend guard, Spot watcher)

- Context grows `1024 -> 4096 -> 8192`; tokens per optimizer step stay constant, so the learning-rate schedule is simply a function of the step.
- A checkpoint is written every `checkpoint_every_minutes`, at every stage end, on **Stop**, on a **Spot interruption notice**, and when the budget guard fires. Each save is written to a temp file and renamed (never corrupts the last good one) and copied to S3 if `S3_URI` is set.
- The guard stops the run when *estimated GPU spend* (`GPU-hours x PRICE_PER_HOUR`) reaches `TRAIN_BUDGET_USD`.

In [ ]:
last_ckpt_time = time.time()


def spent_usd():
    return train_state["gpu_seconds"] / 3600.0 * PRICE_PER_HOUR


def save_checkpoint(filename, loss=None, upload=True):
    """Model + optimizer + counters + dataset position. Written to a .tmp file, then renamed."""
    global last_ckpt_time
    os.makedirs(config.save_dir, exist_ok=True)
    final_path = os.path.join(config.save_dir, filename)
    tmp_path = final_path + ".tmp"
    torch.save({
        "global_step": train_state["global_step"],
        "docs_seen": train_state["docs_seen"],
        "gpu_seconds": train_state["gpu_seconds"],
        "data_state": train_state["data_state"],
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "loss_log": loss_log,
        "loss": loss,
    }, tmp_path)
    os.replace(tmp_path, final_path)
    record_progress(filename, {"global_step": train_state["global_step"], "docs_seen": train_state["docs_seen"],
                               "gpu_hours": round(train_state["gpu_seconds"] / 3600, 3),
                               "spent_usd_est": round(spent_usd(), 2)})
    if upload:
        s3_upload(final_path)
        s3_upload(PROGRESS_FILE)
    last_ckpt_time = time.time()
    return final_path


# ---- Spot interruption watcher: AWS gives a 2-minute warning through the instance metadata service ----
SPOT_NOTICE = threading.Event()


def _spot_watcher():
    import urllib.request, urllib.error
    base = "http://169.254.169.254/latest"
    while True:
        try:
            req = urllib.request.Request(f"{base}/api/token", method="PUT",
                                         headers={"X-aws-ec2-metadata-token-ttl-seconds": "300"})
            token = urllib.request.urlopen(req, timeout=2).read().decode()
        except Exception:
            return                                       # not running on EC2 -> nothing to watch
        try:
            req = urllib.request.Request(f"{base}/meta-data/spot/instance-action", headers={"X-aws-ec2-metadata-token": token})
            urllib.request.urlopen(req, timeout=2)       # 200 = a stop/terminate is coming; 404 = all clear
            SPOT_NOTICE.set()
            return
        except urllib.error.HTTPError:
            pass
        except Exception:
            pass
        time.sleep(5)


threading.Thread(target=_spot_watcher, daemon=True).start()


def train_stage(seq_len, end_step):
    """Train at one context length until global_step == end_step. Returns 'done', 'spot', 'budget' or 'data_end'."""
    micro_batch = config.micro_tokens // seq_len
    accum = config.tokens_per_step // config.micro_tokens
    model.grad_ckpt = bool(plan["grad_ckpt"].get(str(seq_len), False))

    state0 = train_state["data_state"]
    stream = PackedStream(
        config.dataset_mixture, tokenizer, seq_len, micro_batch, BOS_ID, EOS_ID,
        data_state=state0, skip_docs=train_state["docs_seen"] if state0 is None else 0,
        max_doc_chars=config.max_doc_chars,
    )
    docs_base = train_state["docs_seen"]
    it = iter(stream)
    model.train()
    status = "done"
    bar = tqdm(total=plan["total_steps"], initial=train_state["global_step"], desc=f"ctx {seq_len}")
    try:
        while train_state["global_step"] < end_step:
            if SPOT_NOTICE.is_set():
                status = "spot"
                break
            if spent_usd() >= TRAIN_BUDGET_USD:
                status = "budget"
                break

            t0 = time.time()
            step = train_state["global_step"]
            lr = lr_at(step)
            for g in optimizer.param_groups:
                g["lr"] = lr

            loss_acc = torch.zeros((), device=device)
            last_docs, last_state = None, None
            for _ in range(accum):
                item = next(it, None)
                if item is None:
                    status = "data_end"
                    break
                tokens, last_docs, last_state = item
                tokens = tokens.to(device, non_blocking=True)
                with torch.autocast(device.type, dtype=torch.bfloat16, enabled=USE_BF16):
                    hidden = model(tokens[:, :-1], return_hidden=True)
                    loss = chunked_lm_loss(hidden, model.lm_head, tokens[:, 1:])
                (loss / accum).backward()
                loss_acc += loss.detach() / accum
            if status == "data_end":
                optimizer.zero_grad(set_to_none=True)
                break

            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), config.gradient_clip)
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)

            loss_val = loss_acc.item()                         # also synchronises, so dt below is the real step time
            dt = time.time() - t0
            train_state["global_step"] += 1
            train_state["gpu_seconds"] += dt
            train_state["docs_seen"] = docs_base + last_docs
            train_state["data_state"] = last_state
            bar.update(1)

            if train_state["global_step"] % 10 == 0:
                loss_log.append((train_state["global_step"], loss_val))
                bar.set_postfix({"loss": f"{loss_val:.3f}", "lr": f"{lr:.1e}", "tok/s": f"{config.tokens_per_step / dt:,.0f}",
                                 "gn": f"{float(grad_norm):.2f}", "$": f"{spent_usd():.1f}"})

            if time.time() - last_ckpt_time >= config.checkpoint_every_minutes * 60:
                save_checkpoint("checkpoint_latest.pt", loss=loss_val)
    finally:
        stream.close()
        bar.close()
    return status


print("Training functions defined!")

In [ ]:
# ======================== RUN TRAINING ========================
os.makedirs(config.save_dir, exist_ok=True)
save_config_json()
print("=" * 64)
print(f"Training {config.model_name} on {device} | steps {train_state['global_step']:,} -> {plan['total_steps']:,}")
print(f"Estimated spend so far: ${spent_usd():.2f} of ${TRAIN_BUDGET_USD:.0f} training budget")
print("=" * 64)

finished = False
stop_reason = None
try:
    for st in plan["stage_plan"]:
        if train_state["global_step"] >= st["end_step"]:
            continue
        print(f"\n--- context {st['seq_len']} tokens: step {train_state['global_step']:,} -> {st['end_step']:,} ---")
        status = train_stage(st["seq_len"], st["end_step"])
        if status == "done":
            save_checkpoint("checkpoint_latest.pt")            # stage boundary: cheap insurance
            continue
        stop_reason = status
        break
    else:
        finished = True
except KeyboardInterrupt:
    # You pressed Stop: save right now so nothing since the last periodic checkpoint is lost.
    save_checkpoint("checkpoint_latest.pt")
    print(f"\nStopped by you -- checkpoint saved at step {train_state['global_step']:,}. Run the notebook again to continue.")
    raise

if stop_reason == "spot":
    save_checkpoint("checkpoint_latest.pt")
    print("\nSpot interruption notice received -> checkpoint saved (and copied to S3 if configured). "
          "Launch a new instance, set the same S3_URI and Run all to continue.")
elif stop_reason == "budget":
    save_checkpoint("checkpoint_latest.pt")
    print(f"\nBudget guard: estimated GPU spend ${spent_usd():.2f} reached ${TRAIN_BUDGET_USD:.0f}. Checkpoint saved at step "
          f"{train_state['global_step']:,} / {plan['total_steps']:,}. The model is usable as it is; raise BUDGET_USD to keep training.")
elif stop_reason == "data_end":
    save_checkpoint("checkpoint_latest.pt")
    print("\nThe dataset stream ended early. Checkpoint saved.")
elif finished:
    print(f"\nTraining complete: {train_state['global_step']:,} steps, {train_state['docs_seen']:,} documents, "
          f"estimated spend ${spent_usd():.2f}.")

## 12. Save Final Model

In [ ]:
_finished = globals().get("finished", False)
if _finished:
    final_model_path = save_checkpoint("final_model.pt")
    print(f"Full checkpoint (resumable): {final_model_path}")
else:
    print("Training is not finished -> not writing final_model.pt (checkpoint_latest.pt is the resume point). "
          "Saving the current weights for inference anyway.")

# small inference-only file: bf16 weights (~1/6 the size of the full checkpoint)
weights_path = os.path.join(config.save_dir, "model_weights_bf16.pt")
torch.save({k: v.to(torch.bfloat16) if v.is_floating_point() else v for k, v in model.state_dict().items()}, weights_path)
s3_upload(weights_path)
save_config_json()
for _f in ["config.json"]:
    s3_upload(os.path.join(config.save_dir, _f))
if S3_URI and shutil.which("aws"):
    subprocess.run(["aws", "s3", "sync", os.path.join(config.save_dir, "tokenizer"), S3_URI.rstrip("/") + "/tokenizer", "--only-show-errors"])
print(f"Inference weights: {weights_path}\nConfig + tokenizer saved in {config.save_dir}")

if SHUTDOWN_WHEN_DONE and (_finished or globals().get("stop_reason") == "budget"):
    print("SHUTDOWN_WHEN_DONE=True -> shutting the instance down in 60 seconds (Ctrl-C / interrupt to cancel).")
    time.sleep(60)
    subprocess.run(["sudo", "shutdown", "-h", "now"])

## 13. Long-text generation

`write()` generates up to `max_new_tokens` tokens with the KV cache (a 110M model produces roughly 50-100 tokens/s on an A10G), prints the word count, and optionally saves the text. With a 32k vocabulary, **8000 tokens is about 6000 words**. Because it uses a sliding window, you can ask for even more than the trained 8192-token context.

In [ ]:
model.eval()


def write(prompt, max_new_tokens=8000, temperature=0.8, top_k=50, top_p=0.95,
          repetition_penalty=1.1, repetition_window=256, stop_at_eos=False, save_to=None, show=True):
    ids = torch.tensor([[BOS_ID] + tokenizer.encode(prompt)], dtype=torch.long, device=device)
    t0 = time.time()
    out = model.generate(ids, max_new_tokens=max_new_tokens, temperature=temperature, top_k=top_k, top_p=top_p,
                         repetition_penalty=repetition_penalty, repetition_window=repetition_window,
                         eos_id=EOS_ID if stop_at_eos else None)
    dt = time.time() - t0
    text = tokenizer.decode(out[0].tolist())
    n_new = out.shape[1] - ids.shape[1]
    print(f"[{n_new:,} new tokens, {len(text.split()):,} words in total, {n_new / dt:.0f} tokens/s]")
    if save_to:
        with open(save_to, "w") as f:
            f.write(text)
        print(f"saved to {save_to}")
    if show:
        print(text)
    return text


test_prompts = [
    "The future of artificial intelligence",
    "Machine learning is",
    "The capital of nigeria",
    "2 + 2",
    "def sum",
]
print("=" * 60)
print("BrainiacLLM - short samples")
print("=" * 60)
for p in test_prompts:
    print(f"\nPrompt: {p}")
    print(write(p, max_new_tokens=60, show=False)[:400])

In [ ]:
# ---- a LONG generation (about 6000 words). Takes a few minutes. Change the prompt freely. ----
story = write("Chapter 1\n\nThe village of Ilaje woke slowly that morning, and", max_new_tokens=8000,
              save_to=os.path.join(config.save_dir, "long_sample.txt"), show=False)
print(story[:1500], "\n ...\n", story[-800:])

## 14. Training Summary

In [ ]:
import matplotlib.pyplot as plt

if loss_log:
    steps_, losses_ = zip(*loss_log)
    plt.figure(figsize=(10, 6))
    plt.plot(steps_, losses_, linewidth=1.5)
    for st in plan["stage_plan"][:-1]:
        plt.axvline(st["end_step"], color="gray", linestyle="--", alpha=0.6)
    plt.xlabel("optimizer step (dashed lines = context-length changes)")
    plt.ylabel("training loss")
    plt.title(f"{config.model_name} - training loss")
    plt.grid(True, alpha=0.3)
    plt.savefig(os.path.join(config.save_dir, "training_loss.png"), dpi=120)
    plt.show()

print(f"\n{config.model_name} summary")
print(f"  parameters      : {total_params:,}")
print(f"  architecture    : {config.num_layers} layers, {config.num_heads} heads, {config.embedding_dim} dim, context {config.max_seq_length}")
print(f"  steps           : {train_state['global_step']:,} / {plan['total_steps']:,}")
print(f"  tokens trained  : {train_state['global_step'] * config.tokens_per_step / 1e9:.2f} B")
print(f"  documents seen  : {train_state['docs_seen']:,}")
print(f"  est. GPU spend  : ${spent_usd():.2f}  (budget ${TRAIN_BUDGET_USD:.0f} + ${RESERVE_USD:.0f} reserve)")
if loss_log:
    print(f"  last loss       : {loss_log[-1][1]:.3f}")

## 15. Load the trained model later (for generation only)

In [ ]:
def load_brainiac_model(model_dir, device, filename="model_weights_bf16.pt"):
    """Load config + tokenizer + weights from a saved folder. Works with the small bf16 file or a full checkpoint."""
    with open(os.path.join(model_dir, "config.json")) as f:
        config_dict = json.load(f)
    loaded_config = Config()
    for key, value in config_dict.items():
        setattr(loaded_config, key, value)
    m = BrainiacLLM(loaded_config)
    raw = torch_load_cpu(os.path.join(model_dir, filename))
    state = raw["model_state_dict"] if isinstance(raw, dict) and "model_state_dict" in raw else raw
    m.load_state_dict({k: v.float() if v.is_floating_point() else v for k, v in state.items()})
    m = m.to(device).eval()
    tok = ByteBPETokenizer.load(os.path.join(model_dir, "tokenizer"))
    return m, tok, loaded_config


print("load_brainiac_model() defined -- e.g.:")
print("  model, tok, cfg = load_brainiac_model(config.save_dir, device)")

## 16. Continue / extend training

- **After a Spot interruption, disconnect or reboot:** start the instance (or a new one with the same `S3_URI`), then **Run all**. The notebook restores weights, optimizer, step, dataset position and `run_plan.json` and carries on; the progress bar shows the position in the whole run.
- **Budget guard stopped it early** (spend reached the limit before the last step): the weights are still a usable model. To keep going, raise `BUDGET_USD` and Run all; the learning rate continues from where it stopped (it will not have finished its decay).
- **Want more tokens in total?** Delete `run_plan.json`, raise `BUDGET_USD`, and start a fresh run, because the cosine schedule is tied to the planned length.

Keep `config`, `tokens_per_step`, `micro_tokens` and the stage list identical between sessions of the same run.